# ECG Clinical Classification with PTB-XL

This notebook extends the baseline MIT-BIH heartbeat classification pipeline into a more clinically realistic ECG classification task using the PTB-XL dataset.

The notebook is designed to run both locally and on Google Colab.

## Initial Configuration

In [ ]:
# Install missing packages only when running on Google Colab

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q wfdb tqdm

In [ ]:
# Imports

from pathlib import Path
import random
import numpy as np

import matplotlib.pyplot as plt
import pandas as pd
import os

from tqdm.auto import tqdm
import wfdb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Project paths
# If the notebook is inside a notebooks/ folder, the project root is the parent folder.

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

MODELS_DIR = PROJECT_ROOT / "models"
CACHE_DIR = PROJECT_ROOT / "cache"
DATA_DIR = PROJECT_ROOT / "data" / "ptb-xl"

MODELS_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Cache directory:", CACHE_DIR)

In [ ]:
# Reproducibility and device selection

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device used:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected.")

## 1. PTB-XL Dataset Preprocessing

Compared to MIT-BIH, PTB-XL contains full 12-lead ECG recordings and clinical metadata such as age and sex.

In [ ]:
metadata_path = os.path.join(DATA_DIR, "ptbxl_database.csv")

metadata_df = pd.read_csv(metadata_path)

print(metadata_df.shape)
metadata_df.head()

This csv file contains the mappings from code to Diagnostic Class

In [ ]:
scp_statements_path = os.path.join(DATA_DIR, "scp_statements.csv")
scp_df = pd.read_csv(scp_statements_path)
scp_df.head()

In [ ]:
import ast
metadata_df["scp_codes"] = metadata_df["scp_codes"].apply(ast.literal_eval)

In [ ]:
superclass_map = {}

for _, row in scp_df.iterrows():
    
    code = row["Unnamed: 0"]
    
    if row["diagnostic"] == 1:
        superclass_map[code] = row["diagnostic_class"]

In [ ]:
def extract_superclass(scp_codes):
    labels = []

    for code in scp_codes.keys():
        if code in superclass_map:
            labels.append(superclass_map[code])

    labels = list(set(labels))

    if len(labels) == 0:
        return None

    return labels[0]


metadata_df["label"] = metadata_df["scp_codes"].apply(extract_superclass)

In [ ]:
metadata_df = metadata_df.dropna(subset=["label"])

metadata_df["label"].value_counts()

Here, we encode the labels

In [ ]:
label_map = {
    "NORM": 0,
    "MI": 1,
    "STTC": 2,
    "CD": 3,
    "HYP": 4,
}

metadata_df["target"] = metadata_df["label"].map(label_map)

Load ECG

In [ ]:
sample_path = os.path.join(DATA_DIR, metadata_df.iloc[0]["filename_lr"])
signal, meta = wfdb.rdsamp(sample_path)
print(signal.shape)

Plot for visualizing

In [ ]:
plt.figure(figsize=(15, 8))

for i in range(12):
    plt.plot(signal[:, i] + i * 3)

plt.title("12-lead ECG example")
plt.xlabel("Time")
plt.ylabel("Amplitude")

plt.show()

We deal with missing values

In [ ]:
metadata_df["age"] = metadata_df["age"].fillna(metadata_df["age"].median())

age_mean = metadata_df["age"].mean()
age_std = metadata_df["age"].std()

metadata_df["age"] = (
    metadata_df["age"] - age_mean
) / age_std

In [ ]:
metadata_df["sex"] = metadata_df["sex"].fillna(0)

We create the tensors now before training

In [ ]:
all_ecg = []
all_meta = []
all_labels = []

for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):

    signal_path = os.path.join(
        DATA_DIR,
        row["filename_lr"]
    )

    signal, _ = wfdb.rdsamp(signal_path)

    signal = signal.T.astype(np.float32)

    # Per-lead normalization
    signal = (
        signal - signal.mean(axis=1, keepdims=True)
    ) / (
        signal.std(axis=1, keepdims=True) + 1e-8
    )

    metadata = np.array([
        row["age"],
        row["sex"],
    ], dtype=np.float32)

    label = np.int64(row["target"])

    all_ecg.append(signal)
    all_meta.append(metadata)
    all_labels.append(label)

In [ ]:
X_ecg = torch.tensor(
    np.array(all_ecg),
    dtype=torch.float32
)

X_meta = torch.tensor(
    np.array(all_meta),
    dtype=torch.float32
)

y = torch.tensor(
    np.array(all_labels),
    dtype=torch.long
)

## 2. Train / validation / test split

In [ ]:
def stratified_train_val_test_split(X_ecg, X_meta, y, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Stratified split without scikit-learn.

    Each class is split separately, then shuffled.
    """

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    generator = torch.Generator().manual_seed(seed)

    train_indices = []
    val_indices = []
    test_indices = []

    for class_id in torch.unique(y):
        class_indices = torch.where(y == class_id)[0]
        shuffled = class_indices[torch.randperm(len(class_indices), generator=generator)]

        n_total = len(shuffled)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)

        train_indices.append(shuffled[:n_train])
        val_indices.append(shuffled[n_train:n_train + n_val])
        test_indices.append(shuffled[n_train + n_val:])

    train_indices = torch.cat(train_indices)
    val_indices = torch.cat(val_indices)
    test_indices = torch.cat(test_indices)

    train_indices = train_indices[torch.randperm(len(train_indices), generator=generator)]
    val_indices = val_indices[torch.randperm(len(val_indices), generator=generator)]
    test_indices = test_indices[torch.randperm(len(test_indices), generator=generator)]

    X_ecg_train = X_ecg[train_indices]
    X_meta_train = X_meta[train_indices]
    y_train = y[train_indices]

    X_ecg_val = X_ecg[val_indices]
    X_meta_val = X_meta[val_indices]
    y_val = y[val_indices]

    X_ecg_test = X_ecg[test_indices]
    X_meta_test = X_meta[test_indices]
    y_test = y[test_indices]

    return (X_ecg_train, X_ecg_val, X_ecg_test, X_meta_train, X_meta_val, X_meta_test, y_train, y_val, y_test)


X_ecg_train, X_ecg_val, X_ecg_test, X_meta_train, X_meta_val, X_meta_test, y_train, y_val, y_test = stratified_train_val_test_split(
    X_ecg,
    X_meta,
    y,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=SEED,
)

See the distribution

In [ ]:
class_names = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP",
]

num_classes = len(class_names)

def count_classes(labels, num_classes):

    counts = torch.zeros(num_classes, dtype=torch.int64)

    for class_id in range(num_classes):
        counts[class_id] = (labels == class_id).sum()

    return counts

for split_name, split_labels in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:

    print(f"\n{split_name} distribution:")

    split_counts = count_classes(
        split_labels,
        num_classes
    )

    for class_id in range(num_classes):

        count = split_counts[class_id].item()

        percentage = (
            100 * count / len(split_labels)
        )

        print(
            f"{class_id} - "
            f"{class_names[class_id]}: "
            f"{count} samples "
            f"({percentage:.2f}%)"
        )